# SurfaceTarget-CRC — phase 01 — patch 1 20

Extracted from the original Google Colab research notebook.

**Note:** Earlier patches and project data may be required to execute these cells.


In [ ]:
# ============================================================
# SurfaceTarget-CRC
# PHASE 1 — PATCH 1.20 (CORRECTED)
#
# EXPLORATORY COAD DIFFERENTIAL EXPRESSION
#
# Discovery comparison:
# - Tier B COAD primary tumors
# - Patient-independent COAD normal tissues
# - Shared GDC STAR workflow version
#
# Additional evidence:
# - Tier A COAD tumor expression support
#
# FIX:
# Explicit Pandas string indexes for AnnData compatibility.
#
# IMPORTANT:
# 1. Tier B is computationally low-MSI, not verified MSS.
# 2. No sample is removed from the source dataset.
# 3. Patient overlap is avoided in the discovery contrast.
# 4. Results are exploratory pending sensitivity analyses.
# 5. Differential expression is not therapeutic validation.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version, PackageNotFoundError

import subprocess
import sys
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

ROOT = Path("/content/SurfaceTarget-CRC")

META = ROOT / "data/metadata"
PROCESSED = ROOT / "data/processed"
RESULTS = ROOT / "results/tables"
FIGURES = ROOT / "results/figures"

RESULTS.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

MATRIX_PATH = (
    PROCESSED / "tcga_unstranded_raw_counts.npz"
)

SAMPLES_PATH = (
    META / "tcga_expression_sample_metadata.csv"
)

GENES_PATH = (
    META / "tcga_gene_annotations.csv"
)

QC_PATH = (
    RESULTS / "tcga_full_cohort_qc.csv"
)

print("=" * 65)
print("SURFACETARGET-CRC")
print("PATCH 1.20 — EXPLORATORY COAD DIFFERENTIAL EXPRESSION")
print("=" * 65)


# ------------------------------------------------------------
# 2. Check PyDESeq2 installation
# ------------------------------------------------------------

print("\nChecking differential expression dependencies...")

REQUIRED_VERSION = "0.5.4"

try:
    installed_version = version("pydeseq2")

except PackageNotFoundError:
    installed_version = None

if installed_version != REQUIRED_VERSION:

    print(
        "Installing PyDESeq2",
        REQUIRED_VERSION
    )

    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        f"pydeseq2=={REQUIRED_VERSION}"
    ])

from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

print("\nDependency versions:")

print("PyDESeq2:", version("pydeseq2"))
print("Pandas:", pd.__version__)
print("AnnData:", version("anndata"))
print("NumPy:", np.__version__)


# ------------------------------------------------------------
# 3. Load validated data
# ------------------------------------------------------------

required_paths = [
    MATRIX_PATH,
    SAMPLES_PATH,
    GENES_PATH,
    QC_PATH
]

for path in required_paths:

    if not path.exists():

        raise FileNotFoundError(
            f"Required project file missing: {path}"
        )

with np.load(
    MATRIX_PATH,
    allow_pickle=False
) as archive:

    counts = archive["counts"]
    gene_ids = archive["gene_ids"]
    file_ids = archive["file_ids"]

samples = pd.read_csv(
    SAMPLES_PATH,
    dtype=str,
    keep_default_na=False
)

genes = pd.read_csv(
    GENES_PATH,
    dtype=str,
    keep_default_na=False
)

qc = pd.read_csv(
    QC_PATH,
    dtype=str,
    keep_default_na=False
)

print("\nExpression data loaded.")

print("Matrix:", counts.shape)
print("Samples:", len(samples))
print("Gene annotations:", len(genes))


# ------------------------------------------------------------
# 4. Validate sample and gene alignment
# ------------------------------------------------------------

if counts.shape != (60660, 472):

    raise RuntimeError(
        "Unexpected count matrix dimensions."
    )

if not np.array_equal(
    file_ids,
    samples["file_id"].to_numpy()
):

    raise RuntimeError(
        "Sample metadata alignment failed."
    )

if not np.array_equal(
    file_ids,
    qc["file_id"].to_numpy()
):

    raise RuntimeError(
        "QC metadata alignment failed."
    )

if not np.array_equal(
    gene_ids,
    genes["gene_id"].to_numpy()
):

    raise RuntimeError(
        "Gene annotation alignment failed."
    )

if not np.issubdtype(
    counts.dtype,
    np.integer
):

    raise RuntimeError(
        "Raw count matrix must be integer-valued."
    )

if np.any(counts < 0):

    raise RuntimeError(
        "Negative raw counts detected."
    )

print("\nData alignment validation: PASSED")


# ------------------------------------------------------------
# 5. Identify shared processing workflow
# ------------------------------------------------------------

normal_versions = samples.loc[
    samples["dataset_group"].eq("Normal"),
    "workflow_version"
].unique()

if len(normal_versions) != 1:

    raise RuntimeError(
        "Normal samples do not share one workflow."
    )

SHARED_WORKFLOW = normal_versions[0]

print("\nDiscovery workflow:")
print(SHARED_WORKFLOW)


# ------------------------------------------------------------
# 6. Define COAD discovery populations
# ------------------------------------------------------------

coad = samples["project"].eq("TCGA-COAD")

shared = samples["workflow_version"].eq(
    SHARED_WORKFLOW
)

tier_b_mask = (
    coad
    & shared
    & samples["dataset_group"].eq("Tumor")
    & samples["cohort_tier"].eq("Tier B")
)

normal_mask = (
    coad
    & shared
    & samples["dataset_group"].eq("Normal")
)

tier_a_mask = (
    coad
    & samples["dataset_group"].eq("Tumor")
    & samples["cohort_tier"].eq("Tier A")
)

tumor_patient_ids = set(
    samples.loc[
        tier_b_mask,
        "patient_id"
    ]
)

# Avoid including tumor and normal measurements
# from the same patient in the initial
# unpaired differential expression model.

overlapping_normals = (
    normal_mask
    & samples["patient_id"].isin(
        tumor_patient_ids
    )
)

independent_normal_mask = (
    normal_mask
    & ~samples["patient_id"].isin(
        tumor_patient_ids
    )
)

tumor_indices = np.flatnonzero(
    tier_b_mask.to_numpy()
)

normal_indices = np.flatnonzero(
    independent_normal_mask.to_numpy()
)

tier_a_indices = np.flatnonzero(
    tier_a_mask.to_numpy()
)

print("\n" + "-" * 65)
print("DISCOVERY COHORT DEFINITION")
print("-" * 65)

print("\nTier B COAD tumors:", len(tumor_indices))

print(
    "Available COAD normals:",
    int(normal_mask.sum())
)

print(
    "Normals overlapping Tier B patients:",
    int(overlapping_normals.sum())
)

print(
    "Patient-independent normals:",
    len(normal_indices)
)

print(
    "Tier A COAD expression-support samples:",
    len(tier_a_indices)
)

if len(tumor_indices) != 194:

    raise RuntimeError(
        "Unexpected Tier B tumor count."
    )

if len(tier_a_indices) != 98:

    raise RuntimeError(
        "Unexpected Tier A tumor count."
    )

if len(normal_indices) < 20:

    raise RuntimeError(
        "Insufficient independent normal samples."
    )

normal_patients = set(
    samples.iloc[normal_indices]["patient_id"]
)

if normal_patients & tumor_patient_ids:

    raise RuntimeError(
        "Patient overlap remains in discovery cohort."
    )

selected_indices = np.concatenate([
    normal_indices,
    tumor_indices
])

selected_samples = samples.iloc[
    selected_indices
].copy()

if selected_samples["patient_id"].duplicated().any():

    raise RuntimeError(
        "Repeated patient in discovery cohort."
    )

if selected_samples["workflow_version"].nunique() != 1:

    raise RuntimeError(
        "Discovery workflow is not consistent."
    )

print("\nPatient independence: PASSED")
print("Workflow restriction: PASSED")


# ------------------------------------------------------------
# 7. Save cohort selection metadata
# ------------------------------------------------------------

selection = selected_samples.copy()

selection["condition"] = (
    ["Normal"] * len(normal_indices)
    + ["Tumor"] * len(tumor_indices)
)

selection["analysis"] = (
    "COAD_TierB_vs_independent_normal"
)

selection["qc_review_flags"] = (
    qc.iloc[selected_indices][
        "n_qc_review_flags"
    ].to_numpy()
)

selection.to_csv(
    META / "tcga_coad_de_discovery_samples.csv",
    index=False
)

print("\nDiscovery sample manifest saved.")


# ------------------------------------------------------------
# 8. Extract protein-coding raw counts
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PREPARING DIFFERENTIAL EXPRESSION MATRIX")
print("-" * 65)

coding_mask = (
    genes["gene_type"]
    .eq("protein_coding")
    .to_numpy(dtype=bool)
)

coding_gene_ids = gene_ids[coding_mask]

# PyDESeq2 expects samples x genes.

analysis_counts = (
    counts[coding_mask, :][
        :, selected_indices
    ].T
)

if analysis_counts.shape[0] != len(
    selected_indices
):

    raise RuntimeError(
        "DE count matrix sample count mismatch."
    )


# ------------------------------------------------------------
# 9. Pre-filter very low-expression genes
# ------------------------------------------------------------

# CPM is used ONLY for expression filtering.
# The statistical model receives original
# integer raw counts.

total_library_sizes = counts[
    :, selected_indices
].sum(axis=0)

if np.any(total_library_sizes <= 0):

    raise RuntimeError(
        "Zero assigned-count library detected."
    )

cpm = (
    analysis_counts.astype(np.float64)
    / total_library_sizes[:, None]
    * 1_000_000
)

minimum_samples = max(
    10,
    int(np.ceil(
        0.10 * len(selected_indices)
    ))
)

expressed_mask = (
    np.count_nonzero(
        cpm >= 1,
        axis=0
    )
    >= minimum_samples
)

selected_gene_ids = (
    coding_gene_ids[expressed_mask]
)

filtered_counts = (
    analysis_counts[:, expressed_mask]
)

print(
    "\nOriginal protein-coding genes:",
    len(coding_gene_ids)
)

print(
    "Genes passing expression filter:",
    len(selected_gene_ids)
)

print(
    "Minimum samples with CPM >= 1:",
    minimum_samples
)

if len(selected_gene_ids) < 5000:

    raise RuntimeError(
        "Unexpectedly few genes passed filtering."
    )

if np.any(filtered_counts < 0):

    raise RuntimeError(
        "Negative counts in DE matrix."
    )


# ------------------------------------------------------------
# 10. Construct PyDESeq2 inputs
# ------------------------------------------------------------

selected_file_ids = (
    selected_samples["file_id"].tolist()
)

counts_df = pd.DataFrame(
    filtered_counts,
    index=selected_file_ids,
    columns=selected_gene_ids
)

metadata = pd.DataFrame(
    {
        "condition": (
            ["Normal"] * len(normal_indices)
            + ["Tumor"] * len(tumor_indices)
        )
    },
    index=selected_file_ids
)

metadata["condition"] = pd.Categorical(
    metadata["condition"],
    categories=["Normal", "Tumor"],
    ordered=True
)

if not counts_df.index.equals(metadata.index):

    raise RuntimeError(
        "DE count and metadata indices are misaligned."
    )

if counts_df.index.duplicated().any():

    raise RuntimeError(
        "Duplicate samples in DE input."
    )

print("\nPyDESeq2 input matrix:", counts_df.shape)

print("\nConditions:")

print(
    metadata["condition"]
    .value_counts()
    .to_string()
)


# ============================================================
# 10A. PANDAS / ANNDATA COMPATIBILITY FIX
#
# This explicitly converts the identifiers to
# Pandas StringDtype indexes.
#
# AnnData can then recognize the existing
# string dtype instead of attempting the
# incompatible StringDtype(na_value=...) call.
#
# The gene counts and biological metadata
# are not modified.
# ============================================================

print("\n" + "-" * 65)
print("APPLYING ANNDATA STRING-INDEX FIX")
print("-" * 65)

counts_df.index = pd.Index(
    counts_df.index,
    dtype="string"
)

counts_df.columns = pd.Index(
    counts_df.columns,
    dtype="string"
)

metadata.index = pd.Index(
    metadata.index,
    dtype="string"
)

# Verify that the sample identifiers
# remain aligned after conversion.

if not counts_df.index.equals(metadata.index):

    raise RuntimeError(
        "Sample index alignment failed after conversion."
    )

if not np.array_equal(
    counts_df.index.astype(str).to_numpy(),
    np.asarray(selected_file_ids, dtype=str)
):

    raise RuntimeError(
        "Sample identifiers changed during conversion."
    )

if counts_df.index.dtype != "string":

    raise RuntimeError(
        "Sample index is not Pandas StringDtype."
    )

if counts_df.columns.dtype != "string":

    raise RuntimeError(
        "Gene index is not Pandas StringDtype."
    )

if metadata.index.dtype != "string":

    raise RuntimeError(
        "Metadata index is not Pandas StringDtype."
    )

print("\nSample index conversion: PASSED")
print("Gene index conversion: PASSED")
print("Metadata index conversion: PASSED")
print("Sample alignment: PASSED")

print("\nAnnData compatibility fix applied.")


# ------------------------------------------------------------
# 11. Fit differential expression model
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("FITTING DIFFERENTIAL EXPRESSION MODEL")
print("=" * 65)

print("\nModel: ~ condition")
print("Contrast: Tumor versus Normal")

dds = DeseqDataSet(
    counts=counts_df,
    metadata=metadata,
    design="~condition",
    refit_cooks=True,
    n_cpus=2
)

print("\nDESeq2 dataset initialized successfully.")

dds.deseq2()

print("\nDESeq2 model fitting: COMPLETE")


# ------------------------------------------------------------
# 12. Differential expression statistics
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CALCULATING DIFFERENTIAL EXPRESSION")
print("-" * 65)

stats = DeseqStats(
    dds,
    contrast=[
        "condition",
        "Tumor",
        "Normal"
    ],
    alpha=0.05,
    n_cpus=2
)

stats.summary()

results = (
    stats.results_df
    .copy()
    .rename_axis("gene_id")
    .reset_index()
)

required_result_columns = {
    "gene_id",
    "baseMean",
    "log2FoldChange",
    "pvalue",
    "padj"
}

if not required_result_columns.issubset(
    results.columns
):

    raise RuntimeError(
        "Unexpected PyDESeq2 result format."
    )

if results["gene_id"].duplicated().any():

    raise RuntimeError(
        "Duplicate genes in DE results."
    )

print("\nDifferential expression statistics: COMPLETE")


# ------------------------------------------------------------
# 13. Attach gene annotations
# ------------------------------------------------------------

annotation_subset = genes[
    ["gene_id", "gene_name", "gene_type"]
].copy()

results = results.merge(
    annotation_subset,
    on="gene_id",
    how="left",
    validate="one_to_one"
)

if results["gene_name"].isna().any():

    raise RuntimeError(
        "Some genes have missing annotations."
    )

print("\nGene annotations attached successfully.")


# ------------------------------------------------------------
# 14. Add Tier A expression-support metrics
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CALCULATING TIER A EXPRESSION SUPPORT")
print("-" * 65)

# Evaluate expression prevalence
# separately in Tier A.
#
# This is NOT differential expression
# between Tier A and Tier B.

gene_lookup = {
    gene_id: index
    for index, gene_id in enumerate(gene_ids)
}

selected_gene_rows = np.array([
    gene_lookup[gene_id]
    for gene_id in selected_gene_ids
])


def descriptive_expression(indices):

    selected_raw = counts[
        np.ix_(
            selected_gene_rows,
            indices
        )
    ].astype(np.float64)

    libraries = counts[
        :, indices
    ].sum(axis=0)

    if np.any(libraries <= 0):

        raise RuntimeError(
            "Zero library in expression-support group."
        )

    group_cpm = (
        selected_raw
        / libraries[None, :]
        * 1_000_000
    )

    return pd.DataFrame({

        "gene_id": selected_gene_ids,

        "median_cpm": np.median(
            group_cpm,
            axis=1
        ),

        "fraction_cpm_ge1": np.mean(
            group_cpm >= 1,
            axis=1
        )
    })


tier_a_expression = descriptive_expression(
    tier_a_indices
)

tier_b_expression = descriptive_expression(
    tumor_indices
)

normal_expression = descriptive_expression(
    normal_indices
)


tier_a_expression = tier_a_expression.rename(
    columns={
        "median_cpm": "tier_a_median_cpm",
        "fraction_cpm_ge1": "tier_a_fraction_cpm_ge1"
    }
)

tier_b_expression = tier_b_expression.rename(
    columns={
        "median_cpm": "tier_b_median_cpm",
        "fraction_cpm_ge1": "tier_b_fraction_cpm_ge1"
    }
)

normal_expression = normal_expression.rename(
    columns={
        "median_cpm": "normal_median_cpm",
        "fraction_cpm_ge1": "normal_fraction_cpm_ge1"
    }
)


for expression_table in [
    tier_a_expression,
    tier_b_expression,
    normal_expression
]:

    results = results.merge(
        expression_table,
        on="gene_id",
        how="left",
        validate="one_to_one"
    )

print("\nTier A expression support: COMPLETE")


# ------------------------------------------------------------
# 15. Identify exploratory upregulated genes
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("EXPLORATORY DIFFERENTIAL EXPRESSION RESULTS")
print("=" * 65)

results["tumor_upregulated"] = (
    results["padj"].lt(0.05)
    & results["log2FoldChange"].gt(1)
)

results["tumor_downregulated"] = (
    results["padj"].lt(0.05)
    & results["log2FoldChange"].lt(-1)
)

results["tier_a_expression_supported"] = (
    results["tier_a_fraction_cpm_ge1"].ge(0.50)
    & results["tier_a_median_cpm"].ge(1)
)

# Provisional transcriptomic screening genes.
# These are not validated therapeutic targets.

candidates = results[
    results["tumor_upregulated"]
    & results["tier_a_expression_supported"]
].copy()

candidates = candidates.sort_values(
    [
        "padj",
        "log2FoldChange"
    ],
    ascending=[
        True,
        False
    ]
)

print("\nGenes tested:", len(results))

print(
    "Tumor-upregulated genes "
    "(padj < 0.05, log2FC > 1):",
    int(results["tumor_upregulated"].sum())
)

print(
    "Tumor-downregulated genes "
    "(padj < 0.05, log2FC < -1):",
    int(results["tumor_downregulated"].sum())
)

print(
    "Upregulated genes with Tier A "
    "expression support:",
    len(candidates)
)

print("\nTop 20 exploratory screening genes:")

display_columns = [
    "gene_name",
    "log2FoldChange",
    "padj",
    "tier_a_median_cpm",
    "tier_a_fraction_cpm_ge1",
    "normal_median_cpm"
]

print(
    candidates[display_columns]
    .head(20)
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.4g}"
    )
)


# ------------------------------------------------------------
# 16. Generate exploratory volcano plot
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("GENERATING DIFFERENTIAL EXPRESSION FIGURE")
print("-" * 65)

plot_results = results[
    results["padj"].notna()
    & results["log2FoldChange"].notna()
].copy()

plot_results["neg_log10_padj"] = (
    -np.log10(
        plot_results["padj"].clip(
            lower=np.finfo(float).tiny
        )
    )
)

fig, ax = plt.subplots(
    figsize=(10, 7)
)

ax.scatter(
    plot_results["log2FoldChange"],
    plot_results["neg_log10_padj"],
    s=8,
    alpha=0.45
)

ax.axvline(
    -1,
    linestyle="--",
    linewidth=0.8
)

ax.axvline(
    1,
    linestyle="--",
    linewidth=0.8
)

ax.axhline(
    -np.log10(0.05),
    linestyle="--",
    linewidth=0.8
)

ax.set_xlabel(
    "Log2 fold change: Tier B tumor / normal"
)

ax.set_ylabel(
    "-log10 adjusted p-value"
)

ax.set_title(
    "TCGA-COAD — Exploratory Tumor vs Normal"
)

ax.grid(alpha=0.2)

fig.tight_layout()

figure_path = (
    FIGURES /
    "tcga_coad_tierb_exploratory_volcano.png"
)

fig.savefig(
    figure_path,
    dpi=200,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

print("Volcano plot saved.")


# ------------------------------------------------------------
# 17. Save expression and statistical results
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING DIFFERENTIAL EXPRESSION RESULTS")
print("-" * 65)

all_results_path = (
    RESULTS /
    "tcga_coad_tierb_vs_normal_deseq2_results.csv"
)

candidate_path = (
    RESULTS /
    "tcga_coad_exploratory_upregulated_genes.csv"
)

results.to_csv(
    all_results_path,
    index=False
)

candidates.to_csv(
    candidate_path,
    index=False
)

size_factors = pd.DataFrame({

    "file_id": dds.obs.index.astype(str),

    "size_factor": dds.obs[
        "size_factors"
    ].to_numpy()
})

size_factors.to_csv(
    RESULTS / "tcga_coad_deseq2_size_factors.csv",
    index=False
)

print("\nDifferential expression files saved.")


# ------------------------------------------------------------
# 18. Save analysis provenance
# ------------------------------------------------------------

provenance = {

    "project": "SurfaceTarget-CRC",

    "patch": "1.20",

    "patch_version": "1.20-corrected",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "analysis": (
        "Exploratory COAD Tier B tumor "
        "versus patient-independent normal"
    ),

    "software": "PyDESeq2",

    "software_version": version("pydeseq2"),

    "pandas_version": pd.__version__,

    "anndata_version": version("anndata"),

    "compatibility_fix": (
        "Explicit Pandas StringDtype indexes "
        "for AnnData sample and gene identifiers"
    ),

    "design": "~condition",

    "contrast": [
        "condition",
        "Tumor",
        "Normal"
    ],

    "workflow_version": SHARED_WORKFLOW,

    "tumor_samples": int(
        len(tumor_indices)
    ),

    "normal_samples": int(
        len(normal_indices)
    ),

    "overlapping_normals_withheld": int(
        overlapping_normals.sum()
    ),

    "tier_a_expression_support_samples": int(
        len(tier_a_indices)
    ),

    "count_column": "unstranded",

    "genes_tested": int(
        len(results)
    ),

    "expression_prefilter": {

        "cpm_threshold": 1,

        "minimum_samples": int(
            minimum_samples
        )
    },

    "tumor_upregulated_genes": int(
        results["tumor_upregulated"].sum()
    ),

    "tier_a_supported_screening_genes": int(
        len(candidates)
    ),

    "multiple_testing": (
        "Benjamini-Hochberg adjusted p-values "
        "through PyDESeq2"
    ),

    "qc_flagged_samples_excluded": False,

    "clinical_covariates_adjusted": False,

    "paired_model": False,

    "sensitivity_analysis_completed": False,

    "limitations": [

        "Tier B is not independently confirmed MSS.",

        "Normal tissues may be tumor-adjacent.",

        "Tissue composition may influence "
        "differential expression.",

        "Clinical covariates are not adjusted.",

        "Other technical confounding may remain.",

        "QC-flagged sample sensitivity "
        "analysis remains pending.",

        "A patient-paired analysis remains pending.",

        "Tier A expression support is descriptive.",

        "RNA expression does not establish "
        "protein abundance.",

        "RNA expression does not establish "
        "cell-surface localization.",

        "Screening genes are not validated "
        "therapeutic targets."
    ]
}

with open(
    META /
    "tcga_coad_exploratory_de_provenance.json",
    "w"
) as f:

    json.dump(
        provenance,
        f,
        indent=4
    )

print("\nAnalysis provenance saved.")


# ------------------------------------------------------------
# 19. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 1.20 SUMMARY")
print("=" * 65)

print(
    "\nDiscovery tumor samples:",
    len(tumor_indices)
)

print(
    "Independent normal samples:",
    len(normal_indices)
)

print(
    "Tier A support samples:",
    len(tier_a_indices)
)

print(
    "\nProtein-coding genes tested:",
    len(results)
)

print(
    "Exploratory upregulated genes:",
    int(results["tumor_upregulated"].sum())
)

print(
    "Upregulated genes with Tier A "
    "expression support:",
    len(candidates)
)

print("\nSaved outputs:")

print("1. tcga_coad_de_discovery_samples.csv")

print("2. tcga_coad_tierb_vs_normal_deseq2_results.csv")

print("3. tcga_coad_exploratory_upregulated_genes.csv")

print("4. tcga_coad_deseq2_size_factors.csv")

print("5. tcga_coad_tierb_exploratory_volcano.png")

print("6. tcga_coad_exploratory_de_provenance.json")

print("\nAnnData compatibility fix: APPLIED")

print("No raw counts modified.")

print("No permanent sample exclusions.")

print("\nPATCH 1.20 COMPLETE")

print("=" * 65)

In [ ]:
%pip install -q --upgrade "pandas==2.3.3" #upgraded as the cell above this was not executing